<a href="https://colab.research.google.com/github/heavenmaker024/114-2PL-Repo61271012H/blob/main/HW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84(III).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安裝必要套件
!pip -q install -U gspread gspread_dataframe gradio pandas beautifulsoup4 google-generativeai python-dateutil requests

import os, uuid, json, re
from datetime import datetime as dt
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import get_as_dataframe
from google.auth import default
from google.colab import userdata

# ==========================================
# 1. 系統驗證與設定
# ==========================================
print("🔄 正在驗證與初始化系統...")
try:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("✅ Google Colab 驗證成功")
except Exception as e:
    print(f"❌ 驗證失敗: {e}")

try:
    api_key = userdata.get('Geminiapikey')
    genai.configure(api_key=api_key)
    print("✅ Gemini API 金鑰載入成功")
except Exception as e:
    print("⚠️ 找不到 Gemini API 金鑰，請確保 Secrets 中已設定 'Geminiapikey'")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1IP4cWt9I6b56WN_QI66PecSvj1Pj7-pqJ4cgN87w2a0/edit?usp=sharing"
gsheets = gc.open_by_url(SHEET_URL)
TIMEZONE = "Asia/Taipei"

# ==========================================
# 2. 試算表工作表初始化
# ==========================================
def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="100", cols=str(len(header)+5))
        ws.update(values=[header], range_name="A1")
        return ws

    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update(values=[header], range_name="A1")
    return ws

TASKS_HEADER = ["id","task","status","priority","est_min","start_time","end_time","actual_min","pomodoros","due_date","labels","notes","created_at","updated_at","completed_at","planned_for"]
LOGS_HEADER = ["log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

ws_tasks = ensure_worksheet(gsheets, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(gsheets, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(gsheets, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    for c in header:
        if c not in df.columns: df[c] = ""
    return df[header]

def write_df(ws, df, header):
    if df.empty:
        ws.clear()
        ws.update(values=[header], range_name="A1")
        return
    df_out = df.copy().astype(str)
    ws.clear()
    ws.update(values=[header] + df_out.values.tolist(), range_name="A1")

def refresh_all():
    return read_df(ws_tasks, TASKS_HEADER).copy(), read_df(ws_logs, LOGS_HEADER).copy(), read_df(ws_clips, CLIPS_HEADER).copy()

tasks_df, logs_df, clips_df = refresh_all()

# ==========================================
# 3. 基礎任務邏輯
# ==========================================
def list_task_choices():
    global tasks_df
    if tasks_df.empty: return []
    def row_label(r): return f"[{r['status']}] (P:{r['priority']}) {r['task']} — {r['id']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8], "task": task.strip(), "status": "todo",
        "priority": priority or "M", "est_min": int(est_min) if est_min else 25,
        "start_time": "", "end_time": "", "actual_min": 0, "pomodoros": 0,
        "due_date": due_date or "", "labels": labels or "", "notes": notes or "",
        "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": planned_for or ""
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    choices = list_task_choices()
    return "✅ 已新增任務", tasks_df, gr.update(choices=choices)

# ==========================================
# 🌟 4. 新功能：強化版 BeautifulSoup 課程公告爬蟲
# ==========================================
def get_clean_webpage(url):
    """強化版 BeautifulSoup 網頁清洗函式"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7",
    }

    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    # 自動偵測並修正編碼 (解決部分學校網站 Big5 亂碼問題)
    resp.encoding = resp.apparent_encoding

    soup = BeautifulSoup(resp.text, "html.parser")
    page_title = soup.title.string.strip() if soup.title else "無標題網頁"

    # 1. 大掃除：移除所有雜訊標籤
    for element in soup(["script", "style", "nav", "footer", "header", "aside", "noscript", "iframe", "form"]):
        element.extract()

    # 2. 精準打擊：嘗試尋找主要的內容區塊
    main_content = soup.find('main') or soup.find('article') or soup.find(id=re.compile('(?i)content|main'))

    # 3. 提取文字並整理換行
    if main_content:
        text_content = main_content.get_text(separator='\n', strip=True)
    else:
        text_content = soup.body.get_text(separator='\n', strip=True) if soup.body else soup.get_text(separator='\n', strip=True)

    # 把連續超過兩個以上的換行符號壓縮成一個，節省 AI 閱讀空間
    text_content = re.sub(r'\n+', '\n', text_content)

    return page_title, text_content

def analyze_announcement_with_ai(text_content, url):
    """呼叫 Gemini 判斷是否有作業，並強制輸出 JSON"""
    sys_prompt = """
    你是一位負責幫大學生整理待辦事項的超級助理。
    請閱讀以下從課程網頁抓取下來的內容，並判斷公告中是否包含「作業」、「報告」、「考試」或任何需要繳交的「待辦事項」。

    如果有找到待辦事項，請「嚴格」輸出以下 JSON 格式（不要加上任何
```json 標記，純粹輸出 JSON）：
    {
      "has_task": true,
      "task_name": "[自動產生的簡短任務名稱，例如：作業一、期中報告]",
      "due_date": "[YYYY-MM-DD 格式的截止日期。如果沒有提到確切日期，請留空字串]",
      "priority": "[根據急迫性填寫 H (高), M (中), 或 L (低)]",
      "notes": "[擷取公告中的重點事項、規定或繳交方式，請用 1-2 句話總結]"
    }

    如果完全沒有需要執行的待辦事項（例如純粹的心得分享、一般休假公告），請輸出：
    {
      "has_task": false
    }
    """
    try:
        model = genai.GenerativeModel("gemini-2.5-flash", generation_config={"response_mime_type": "application/json"})
        # 取前 6000 字元，避免超出 Token 限制
        resp = model.generate_content(f"{sys_prompt}\n\n來源網址：{url}\n\n公告內容：\n{text_content[:6000]}")
        return json.loads(resp.text)
    except Exception as e:
        return {"has_task": False, "error": str(e)}

def auto_crawl_course_announcement(url, progress=gr.Progress()):
    """主流程：爬取 -> AI 判斷 -> 自動寫入任務"""
    global tasks_df
    if not url.strip():
        return "⚠️ 請輸入網址", tasks_df, gr.update()

    try:
        # 1. 爬取網頁 (使用強化的 BeautifulSoup)
        progress(0.2, desc="正在讀取並清理網頁內容...")
        page_title, text_content = get_clean_webpage(url)

        if len(text_content) < 20:
            return f"⚠️ 抓取成功，但網頁內容太少無法分析（可能是需要登入的系統）。標題：{page_title}", tasks_df, gr.update()

        # 2. 讓 AI 分析內容
        progress(0.6, desc="網頁清理完成！交給 AI 尋找待辦任務...")
        ai_result = analyze_announcement_with_ai(text_content, url)

        if ai_result.get("error"):
            return f"❌ AI 分析失敗：{ai_result['error']}", tasks_df, gr.update()

        if not ai_result.get("has_task"):
            return f"ℹ️ 掃描完畢。這篇公告沒有找到明確的待辦事項喔！(網頁標題：{page_title})", tasks_df, gr.update()

        # 3. 有待辦任務！寫入系統
        progress(0.9, desc="找到任務！正在寫入 Google Sheet...")
        _now = tznow().isoformat()
        notes_with_source = f"{ai_result.get('notes', '')}\n\n🔗 來源網址：{url}\n📄 網頁標題：{page_title}"

        new_task = pd.DataFrame([{
            "id": str(uuid.uuid4())[:8],
            "task": f"📚 {ai_result.get('task_name', '未命名課程任務')[:110]}",
            "status": "todo",
            "priority": ai_result.get("priority", "H"),
            "est_min": 60,
            "start_time": "", "end_time": "", "actual_min": 0, "pomodoros": 0,
            "due_date": ai_result.get("due_date", ""),
            "labels": "from:AI_course_bot",
            "notes": notes_with_source,
            "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": ""
        }])

        tasks_df = pd.concat([tasks_df, new_task], ignore_index=True)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        choices = list_task_choices()

        success_msg = (
            f"✅ **成功擷取待辦事項！**\n"
            f"- **任務名稱**：{ai_result.get('task_name')}\n"
            f"- **死線日期**：{ai_result.get('due_date') or '未註明'}\n"
            f"- **優先等級**：{ai_result.get('priority')}\n"
            f"*(已自動加入您的 Tasks 任務清單中)*"
        )
        return success_msg, tasks_df, gr.update(choices=choices)

    except requests.exceptions.RequestException as e:
        return f"❌ 網路請求失敗，請檢查網址是否正確或是否需要登入權限：{e}", tasks_df, gr.update()
    except Exception as e:
        return f"❌ 發生未預期的錯誤：{e}", tasks_df, gr.update()

# ==========================================
# 5. Gradio 介面綁定
# ==========================================
with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘（Google Sheet＋Crawler＋AI 計畫）")

    with gr.Tab("Tasks (任務清單)"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）")
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD）")
                btn_add = gr.Button("➕ 手動新增任務", variant="primary")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="目前任務清單", interactive=False)

    with gr.Tab("🎓 課程公告 AI 助手 (Smart Crawler)"):
        gr.Markdown("### 懶得看長篇大論的課程公告？交給 AI 吧！")
        gr.Markdown("貼上學校公告或課程網頁的連結。系統會利用 `BeautifulSoup` 深度過濾網頁雜訊，再交由 AI 抓出「作業、報告、死線」，自動建好待辦任務！")
        with gr.Row():
            course_url_input = gr.Textbox(label="請輸入課程公告網址 (URL)", placeholder="https://...", scale=4)
            btn_course_scan = gr.Button("🤖 自動掃描並加入任務", variant="primary", scale=1)

        msg_course_result = gr.Markdown(label="執行結果")

    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, gr.State(""), gr.State(""), gr.State("")], outputs=[msg_add, grid_tasks])

    btn_course_scan.click(
        auto_crawl_course_announcement,
        inputs=[course_url_input],
        outputs=[msg_course_result, grid_tasks]
    )

demo.launch(debug=True)

🔄 正在驗證與初始化系統...
✅ Google Colab 驗證成功
✅ Gemini API 金鑰載入成功
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8369ddb51c67bd7a49.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8369ddb51c67bd7a49.gradio.live


In [3]:
import os, uuid, json, re
from datetime import datetime as dt
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import get_as_dataframe
from google.auth import default
from google.colab import userdata

# ==========================================
# 1. 系統驗證與設定
# ==========================================
print("🔄 正在驗證與初始化系統...")
try:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("✅ Google Colab 驗證成功")
except Exception as e:
    print(f"❌ 驗證失敗: {e}")

try:
    api_key = userdata.get('Geminiapikey')
    genai.configure(api_key=api_key)
    print("✅ Gemini API 金鑰載入成功")
except Exception as e:
    print("⚠️ 找不到 Gemini API 金鑰，請確保 Secrets 中已設定 'Geminiapikey'")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1IP4cWt9I6b56WN_QI66PecSvj1Pj7-pqJ4cgN87w2a0/edit?usp=sharing"
gsheets = gc.open_by_url(SHEET_URL)
TIMEZONE = "Asia/Taipei"

# ==========================================
# 2. 試算表工作表初始化
# ==========================================
def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="100", cols=str(len(header)+5))
        ws.update(values=[header], range_name="A1")
        return ws

    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update(values=[header], range_name="A1")
    return ws

TASKS_HEADER = ["id","task","status","priority","est_min","start_time","end_time","actual_min","pomodoros","due_date","labels","notes","created_at","updated_at","completed_at","planned_for"]
LOGS_HEADER = ["log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

ws_tasks = ensure_worksheet(gsheets, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(gsheets, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(gsheets, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    for c in header:
        if c not in df.columns: df[c] = ""
    return df[header]

def write_df(ws, df, header):
    if df.empty:
        ws.clear()
        ws.update(values=[header], range_name="A1")
        return
    df_out = df.copy().astype(str)
    ws.clear()
    ws.update(values=[header] + df_out.values.tolist(), range_name="A1")

def refresh_all():
    return read_df(ws_tasks, TASKS_HEADER).copy(), read_df(ws_logs, LOGS_HEADER).copy(), read_df(ws_clips, CLIPS_HEADER).copy()

tasks_df, logs_df, clips_df = refresh_all()

# ==========================================
# 3. 基礎任務邏輯
# ==========================================
def list_task_choices():
    global tasks_df
    if tasks_df.empty: return []
    def row_label(r): return f"[{r['status']}] (P:{r['priority']}) {r['task']} — {r['id']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8], "task": task.strip(), "status": "todo",
        "priority": priority or "M", "est_min": int(est_min) if est_min else 25,
        "start_time": "", "end_time": "", "actual_min": 0, "pomodoros": 0,
        "due_date": due_date or "", "labels": labels or "", "notes": notes or "",
        "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": planned_for or ""
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    choices = list_task_choices()
    return "✅ 已新增任務", tasks_df, gr.update(choices=choices)

# ==========================================
# 🌟 4. 新功能：強化版 BeautifulSoup 課程公告爬蟲
# ==========================================
def get_clean_webpage(url):
    """強化版 BeautifulSoup 網頁清洗函式"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7",
    }

    print(f"Attempting to fetch URL: {url}") # Added print statement for URL

    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    # 自動偵測並修正編碼 (解決部分學校網站 Big5 亂碼問題)
    resp.encoding = resp.apparent_encoding

    soup = BeautifulSoup(resp.text, "html.parser")
    page_title = soup.title.string.strip() if soup.title else "無標題網頁"

    # 1. 大掃除：移除所有雜訊標籤
    for element in soup(["script", "style", "nav", "footer", "header", "aside", "noscript", "iframe", "form"]):
        element.extract()

    # 2. 精準打擊：嘗試尋找主要的內容區塊
    main_content = soup.find('main') or soup.find('article') or soup.find(id=re.compile('(?i)content|main'))

    # 3. 提取文字並整理換行
    if main_content:
        text_content = main_content.get_text(separator='\n', strip=True)
    else:
        text_content = soup.body.get_text(separator='\n', strip=True) if soup.body else soup.get_text(separator='\n', strip=True)

    # 把連續超過兩個以上的換行符號壓縮成一個，節省 AI 閱讀空間
    text_content = re.sub(r'\n+', '\n', text_content)

    print(text_content)

    return page_title, text_content

def analyze_announcement_with_ai(text_content, url):
    """呼叫 Gemini 判斷是否有作業，並強制輸出 JSON"""
    sys_prompt = """
    你是一位專業的教師甄試資訊助理。請閱讀以下從教育局或學校網頁抓取下來的內容，並判斷公告中是否包含「教師甄試」的招考資訊。

如果有找到相關資訊，請「嚴格」輸出以下 JSON 格式（不要加上任何 ```json 標記，純粹輸出 JSON）：

{
"has_announcement": true,
"school_name": "[學校名稱，若無則填寫公告單位]",
"subjects": "[列出招考的科別與正取名額，例如：高中數學(1)、國中國文(1)]",
"registration_period": "[YYYY/MM/DD 格式的報名起迄日期，若只有截止日請註明：至 YYYY/MM/DD 止]",
"exam_date": "[YYYY/MM/DD 格式的考試日期]",
"location": "[考試地點或學校地址]",
"application_link": "[報名網址或參考網址]",
"notes": "[擷取報名資格限制、電話或備取名額等重點事項，1-2 句話總結]"
}

如果內容與教師甄試無關（例如單純的校園新聞、研習活動），請輸出：
{
"has_announcement": false
}
    """
    try:
        model = genai.GenerativeModel("gemini-2.5-flash", generation_config={"response_mime_type": "application/json"})
        # 取前 6000 字元，避免超出 Token 限制
        resp = model.generate_content(f"{sys_prompt}\n\n來源網址：{url}\n\n公告內容：\n{text_content[:6000]}")
        return json.loads(resp.text)
    except Exception as e:
        return {"has_task": False, "error": str(e)}

def auto_crawl_course_announcement(url, progress=gr.Progress()):
    """主流程：爬取 -> AI 判斷 -> 自動寫入任務"""
    global tasks_df
    if not url.strip():
        return "⚠️ 請輸入網址", tasks_df, gr.update()

    try:
        # 1. 爬取網頁 (使用強化的 BeautifulSoup)
        progress(0.2, desc="正在讀取並清理網頁內容...")
        page_title, text_content = get_clean_webpage(url)

        if len(text_content) < 20:
            return f"⚠️ 抓取成功，但網頁內容太少無法分析（可能是需要登入的系統）。標題：{page_title}", tasks_df, gr.update()

        # 2. 讓 AI 分析內容
        progress(0.6, desc="網頁清理完成！交待給 AI 尋找辦任務...")
        ai_result = analyze_announcement_with_ai(text_content, url)

        if ai_result.get("error"):
            return f"❌ AI 分析失敗：{ai_result['error']}", tasks_df, gr.update()

        if not ai_result.get("has_announcement"): # Changed from has_task
            return f"ℹ️ 掃描完畢。這篇公告沒有找到明確的教師甄試資訊喔！(網頁標題：{page_title})", tasks_df, gr.update()

        # 3. 有待辦任務！寫入系統
        progress(0.9, desc="找到教師甄試資訊！正在寫入 Google Sheet...")
        _now = tznow().isoformat()

        # Construct detailed notes with source
        notes_with_source = (
            f"招考學校/單位: {ai_result.get('school_name', 'N/A')}\n"
            f"招考科別: {ai_result.get('subjects', 'N/A')}\n"
            f"報名期間: {ai_result.get('registration_period', 'N/A')}\n"
            f"考試日期: {ai_result.get('exam_date', 'N/A')}\n"
            f"考試地點: {ai_result.get('location', 'N/A')}\n"
            f"報名連結: {ai_result.get('application_link', 'N/A')}\n"
            f"備註: {ai_result.get('notes', '無')}\n\n"
            f"🔗 來源網址：{url}\n"
            f"📄 網頁標題：{page_title}"
        )

        new_task = pd.DataFrame([{
            "id": str(uuid.uuid4())[:8],
            "task": f"👨‍🏫 {ai_result.get('school_name', '未定學校')}: {ai_result.get('subjects', '未定科別')[:80]} 甄試公告",
            "status": "todo",
            "priority": "H", # Defaulting to High priority for recruitment announcements
            "est_min": 120, # Increased estimate for teacher recruitment info
            "start_time": "", "end_time": "", "actual_min": 0, "pomodoros": 0,
            "due_date": ai_result.get("exam_date", ""), # Using exam_date as due_date
            "labels": "from:AI_teacher_recruit_bot", # New label
            "notes": notes_with_source,
            "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": ""
        }])

        tasks_df = pd.concat([tasks_df, new_task], ignore_index=True)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        choices = list_task_choices()

        success_msg = (
            f"✅ **成功擷取教師甄試公告！**\n"
            f"- **學校名稱**：{ai_result.get('school_name', '未註明')}\n"
            f"- **招考科別**：{ai_result.get('subjects', '未註明')}\n"
            f"- **報名期間**：{ai_result.get('registration_period', '未註明')}\n"
            f"- **考試日期**：{ai_result.get('exam_date', '未註明')}\n"
            f"*(已自動加入您的 Tasks 任務清單中)*"
        )
        return success_msg, tasks_df, gr.update(choices=choices)

    except requests.exceptions.RequestException as e:
        return f"❌ 網路請求失敗，請檢查網址是否正確或是否需要登入權限：{e}", tasks_df, gr.update()
    except Exception as e:
        return f"❌ 發生未預期的錯誤：{e}", tasks_df, gr.update()

# ==========================================
# 5. Gradio 介面綁定
# ==========================================
with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘（Google Sheet＋Crawler＋AI 計畫）")

    with gr.Tab("Tasks (任務清單)"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）")
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD）")
                btn_add = gr.Button("➕ 手動新增任務", variant="primary")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="目前任務清單", interactive=False)

    with gr.Tab("🎓 課程公告 AI 助手 (Smart Crawler)"):
        gr.Markdown("### 懶得看長篇大論的課程公告？交給 AI 吧！")
        gr.Markdown("貼上學校公告或課程網頁的連結。系統會利用 `BeautifulSoup` 深度過濾網頁雜訊，再交由 AI 抓出「作業、報告、死線」，自動建好待辦任務！")
        with gr.Row():
            course_url_input = gr.Textbox(label="請輸入課程公告網址 (URL)", placeholder="https://...", scale=4)
            btn_course_scan = gr.Button("🤖 自動掃描並加入任務", variant="primary", scale=1)

        msg_course_result = gr.Markdown(label="執行結果")

    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, gr.State(""), gr.State(""), gr.State("")], outputs=[msg_add, grid_tasks])

    btn_course_scan.click(
        auto_crawl_course_announcement,
        inputs=[course_url_input],
        outputs=[msg_course_result, grid_tasks]
    )

# 【關鍵修正】：加上 share=True 強制產生公開連結
demo.launch(debug=True, share=True)

🔄 正在驗證與初始化系統...
✅ Google Colab 驗證成功
✅ Gemini API 金鑰載入成功
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fd7fd670c8782659d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1891: UserWarning: A function (add_task) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [markdown, dataframe]
    Output values returned:
        ["✅ 已新增任務",          id  task status priority  est_min start_time end_time  actual_min  \
0  12cb82cc   1.0   todo        M     25.0                             0.0   
1  b7180018   1.0   todo        M     25.0                             0.0   
2  3e6b7ec2   2.0   todo        H     60.0                             0.0   
3  4e16efd6   3.0   todo        H     24.0                             0.0   
4  41465ce7   abc   todo        M     25.0                             0.0   
5  fe44c8fa  論文口試   todo        H    119.0                             0.0   

   pomodoros    due_date labels notes                        created_at  \
0        0.0  2026-04-30               2026-04-30T12:07:59.276216+08:00   
1        0.0  2026-04-3

Attempting to fetch URL: https://www.ptt.cc/bbs/studyteacher/M.1778649837.A.B1D.html
作者
ilbnilbn (jk)
看板
studyteacher
標題
[高中] 115國立嘉義高中第一學期第2次教師甄選
時間
Wed May 13 13:23:50 2026
公告本校115學年度第1學期第2次教師甄選簡章
初試: 5 月 31 日(星期日)   下午 2:00 起
複試: 6 月 13 日（星期六）  報到：上午 7:40-8:20
本次招考科別名額：
英文科：專任教師 2名、代理教師2名。
化學科：專任教師1名。
國文科：代理教師1名。
報名期限：115年5月11日(一)~115年5月25日(一)中午12:00止。
有意報名者，請至本校「教師甄選報名系統」【
http://web.jhenggao.com/iTSelection/signin.aspx?s=200303
】報名，
相關報名表件由「報名系統」產製。
--
※ 發信站: 批踢踢實業坊(ptt.cc), 來自: 101.8.225.212 (臺灣)
※ 文章網址:
https://www.ptt.cc/bbs/studyteacher/M.1778649837.A.B1D.html
※ 編輯: ilbnilbn (101.8.225.212 臺灣), 05/13/2026 13:25:19


/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1891: UserWarning: A function (auto_crawl_course_announcement) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [markdown, dataframe]
    Output values returned:
        ["✅ **成功擷取教師甄試公告！**
- **學校名稱**：國立嘉義高中
- **招考科別**：高中英文(專任2)、高中英文(代理2)、高中化學(專任1)、高中國文(代理1)
- **報名期間**：2026/05/11 ~ 2026/05/25
- **考試日期**：2026/05/31 (初試), 2026/06/13 (複試)
*(已自動加入您的 Tasks 任務清單中)*",          id                                               task status  \
0  12cb82cc                                                1.0   todo   
1  b7180018                                                1.0   todo   
2  3e6b7ec2                                                2.0   todo   
3  4e16efd6                                                3.0   todo   
4  41465ce7                                                abc   todo   
5  fe44c8fa                                               論文口試   todo   
6  04

Attempting to fetch URL: https://www.ptt.cc/bbs/studyteacher/M.1778484295.A.82D.html
作者
flyfamily (flyfamily)
看板
studyteacher
標題
[高中] 115嘉義女中教師甄試
時間
Mon May 11 15:24:49 2026
115國立嘉義女中教師甄選
甄選科目:
國文科 2 名
英文科 2 名
數學科 2 名
物理科 1 名
生物科 1 名
體育科 1 名
報名網址：
https://info.cygsh.cy.edu.tw/teacher/index.asp
報名時間至5/13 (三)截止，初試5/23 (六)、複試5/30 (六)
--
※ 發信站: 批踢踢實業坊(ptt.cc), 來自: 163.27.4.212 (臺灣)
※ 文章網址:
https://www.ptt.cc/bbs/studyteacher/M.1778484295.A.82D.html


/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1891: UserWarning: A function (auto_crawl_course_announcement) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [markdown, dataframe]
    Output values returned:
        ["✅ **成功擷取教師甄試公告！**
- **學校名稱**：國立嘉義女中
- **招考科別**：國文(2)、英文(2)、數學(2)、物理(1)、生物(1)、體育(1)
- **報名期間**：至 2026/05/13 止
- **考試日期**：2026/05/23, 2026/05/30
*(已自動加入您的 Tasks 任務清單中)*",          id                                               task status  \
0  12cb82cc                                                1.0   todo   
1  b7180018                                                1.0   todo   
2  3e6b7ec2                                                2.0   todo   
3  4e16efd6                                                3.0   todo   
4  41465ce7                                                abc   todo   
5  fe44c8fa                                               論文口試   todo   
6  049681d0  👨‍🏫 國立嘉義高中: 高中英

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fd7fd670c8782659d8.gradio.live
